# 🚗 Used Car Price Predictor – India

Predict the **resale price (₹)** of a used car from **age, kilometres driven, fuel type and brand**
(plus optional seller type, transmission and owner count).

**Dataset:** [CarDekho Vehicle Dataset (Kaggle)](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)

**Skills covered:** regression · high-cardinality categorical handling · outlier treatment · log-target modelling · model comparison · interactive prediction

**How to run:** `Runtime → Run all`, then scroll to the bottom to use the **prediction form**.

## 1. Setup

In [ ]:
!pip install -q kagglehub ipywidgets

import warnings, glob, os, datetime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
CURRENT_YEAR = datetime.datetime.now().year
print("Current year used for age calculation:", CURRENT_YEAR)

## 2. Load the data

Option A (default): download automatically from Kaggle using `kagglehub` (no API key needed for public datasets).
Option B: if that fails, a file-upload box appears – download **CAR DETAILS FROM CAR DEKHO.csv** from Kaggle and upload it.

In [ ]:
REQUIRED = {"name", "year", "selling_price", "km_driven", "fuel"}

def find_csv(folder):
    for path in sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)):
        try:
            cols = set(pd.read_csv(path, nrows=1).columns)
            if REQUIRED.issubset(cols):
                return path
        except Exception:
            pass
    return None

csv_path = None
try:
    import kagglehub
    folder = kagglehub.dataset_download("nehalbirla/vehicle-dataset-from-cardekho")
    csv_path = find_csv(folder)
    print("Downloaded from Kaggle ->", csv_path)
except Exception as e:
    print("Auto-download failed:", e)

if csv_path is None:
    from google.colab import files
    print("Please upload the CarDekho CSV file:")
    up = files.upload()
    csv_path = list(up.keys())[0]

raw = pd.read_csv(csv_path)
print("Shape:", raw.shape)
raw.head()

## 3. Feature engineering & cleaning

* **age** = current year − manufacturing year
* **brand** = first word of the car name (e.g. *Maruti Swift Dzire VDI* → `Maruti`)
* Rare brands (fewer than 15 listings) are merged into **`Other`** → keeps the one-hot encoding small and stable.

In [ ]:
df = raw.copy()
df.columns = [c.strip().lower() for c in df.columns]
df = df.drop_duplicates().dropna(subset=["name", "year", "selling_price", "km_driven", "fuel"])

df["brand"] = df["name"].str.strip().str.split().str[0].str.title()
df["brand"] = df["brand"].replace({"Land": "Land Rover", "Mini": "Mini"})  # "Land Rover" splits into "Land"
df["age"] = CURRENT_YEAR - df["year"]
df["fuel"] = df["fuel"].str.strip().str.title()

# Optional extra columns (present in the CarDekho files)
EXTRA_CATS = [c for c in ["seller_type", "transmission", "owner"] if c in df.columns]
for c in EXTRA_CATS:
    df[c] = df[c].astype(str).str.strip().str.title()

# --- Handle MANY categories: group rare brands ---
MIN_BRAND_COUNT = 15
counts = df["brand"].value_counts()
rare = counts[counts < MIN_BRAND_COUNT].index
df["brand"] = df["brand"].where(~df["brand"].isin(rare), "Other")

print("Brands after grouping:", df["brand"].nunique())
print(df["brand"].value_counts().head(12))
print("\nFuel types:", df["fuel"].unique().tolist())
df.head()

## 4. Outlier handling

Prices and kilometres are heavily right-skewed (a few ₹1 crore+ luxury cars, odometers that read 2,000,000 km).
These distort the model, so we:

1. Drop impossible values (age ≤ 0 is kept at minimum 1, absurd km, very low prices).
2. Remove outliers on **log-price** and **km driven** using the **IQR rule**.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["selling_price"], bins=60, ax=ax[0]); ax[0].set_title("Price (raw) – skewed")
sns.histplot(np.log1p(df["selling_price"]), bins=60, ax=ax[1], color="green"); ax[1].set_title("log(Price) – near normal")
sns.boxplot(x=df["km_driven"], ax=ax[2]); ax[2].set_title("km driven – outliers")
plt.tight_layout(); plt.show()

In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

before = len(df)

# Sanity filters
df["age"] = df["age"].clip(lower=1)
df = df[(df["km_driven"] > 500) & (df["km_driven"] < 1_000_000) & (df["selling_price"] >= 20_000)]

# IQR on log-price and on km
df["log_price"] = np.log1p(df["selling_price"])
lo, hi = iqr_bounds(df["log_price"], k=2.0)       # k=2 -> gentler, keeps legit luxury cars
df = df[(df["log_price"] >= lo) & (df["log_price"] <= hi)]
_, km_hi = iqr_bounds(df["km_driven"], k=2.0)
df = df[df["km_driven"] <= km_hi]

print(f"Rows: {before} -> {len(df)}  (removed {before-len(df)} outliers / bad rows)")
print(f"Price range kept : ₹{df['selling_price'].min():,.0f} – ₹{df['selling_price'].max():,.0f}")
print(f"km range kept    : {df['km_driven'].min():,.0f} – {df['km_driven'].max():,.0f}")

## 5. Quick exploratory analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))

sns.scatterplot(data=df, x="age", y="selling_price", alpha=0.3, ax=ax[0, 0])
ax[0, 0].set_title("Price vs Age")

sns.scatterplot(data=df, x="km_driven", y="selling_price", alpha=0.3, ax=ax[0, 1], color="orange")
ax[0, 1].set_title("Price vs Kilometres")

sns.boxplot(data=df, x="fuel", y="selling_price", ax=ax[1, 0])
ax[1, 0].set_title("Price by Fuel type")

top = df.groupby("brand")["selling_price"].median().sort_values(ascending=False).head(12)
sns.barplot(x=top.values, y=top.index, ax=ax[1, 1], color="teal")
ax[1, 1].set_title("Median price by brand (top 12)")

plt.tight_layout(); plt.show()

## 6. Train / test split & preprocessing pipeline

* **Numeric:** `age`, `km_driven` → standardised (needed for Ridge; harmless for trees).
* **Categorical:** `brand`, `fuel` (+ seller type, transmission, owner) → One-Hot Encoding with
  `handle_unknown="infrequent_if_exist"` and `min_frequency`, so **unseen brands at prediction time don't crash** the model.
* **Target:** modelled on **log scale** via `TransformedTargetRegressor`, then converted back to ₹ automatically.

In [ ]:
NUM_FEATURES = ["age", "km_driven"]
CAT_FEATURES = ["brand", "fuel"] + EXTRA_CATS
FEATURES = NUM_FEATURES + CAT_FEATURES

X = df[FEATURES]
y = df["selling_price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                              min_frequency=5, sparse_output=False), CAT_FEATURES),
    ])

def make_model(regressor):
    pipe = Pipeline([("prep", make_preprocessor()), ("reg", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1)

## 7. Compare models (5-fold cross-validation)

In [ ]:
candidates = {
    "Ridge (baseline)": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, reg in candidates.items():
    scores = cross_val_score(make_model(reg), X_train, y_train, cv=kf, scoring="r2", n_jobs=-1)
    cv_rows.append({"Model": name, "CV R² (mean)": scores.mean(), "CV R² (std)": scores.std()})

cv_df = pd.DataFrame(cv_rows).sort_values("CV R² (mean)", ascending=False)
cv_df

## 8. Tune the best tree model

In [ ]:
param_dist = {
    "regressor__reg__learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
    "regressor__reg__max_iter": [200, 300, 500, 700],
    "regressor__reg__max_depth": [None, 4, 6, 8, 10],
    "regressor__reg__min_samples_leaf": [10, 20, 30, 50],
    "regressor__reg__l2_regularization": [0.0, 0.1, 1.0, 5.0],
}

search = RandomizedSearchCV(
    make_model(HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
    param_distributions=param_dist, n_iter=20, cv=3, scoring="r2",
    n_jobs=-1, random_state=RANDOM_STATE, verbose=1,
)
search.fit(X_train, y_train)
print("Best CV R²:", round(search.best_score_, 4))
print("Best params:", search.best_params_)

best_model = search.best_estimator_

## 9. Evaluate on the unseen test set

In [ ]:
pred = best_model.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

print(f"R²   : {r2:.3f}")
print(f"MAE  : ₹{mae:,.0f}")
print(f"RMSE : ₹{rmse:,.0f}")
print(f"MAPE : {mape:.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].scatter(y_test, pred, alpha=0.3)
lims = [0, max(y_test.max(), pred.max())]
ax[0].plot(lims, lims, "r--"); ax[0].set_xlabel("Actual price (₹)"); ax[0].set_ylabel("Predicted price (₹)")
ax[0].set_title("Actual vs Predicted")

sns.histplot((pred - y_test) / 1e5, bins=60, ax=ax[1], color="purple")
ax[1].set_title("Prediction error (₹ lakh)"); ax[1].set_xlabel("Predicted − Actual (lakh)")
plt.tight_layout(); plt.show()

# Prediction interval from log-residuals (used in the input form below)
log_resid = np.log1p(y_test.values) - np.log1p(pred)
RES_LO, RES_HI = np.percentile(log_resid, [10, 90])

## 10. What drives the price? (permutation importance)

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(best_model, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE, scoring="r2")
imp_df = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
imp_df.plot(kind="barh", figsize=(7, 4), color="steelblue", title="Permutation importance (drop in R²)")
plt.tight_layout(); plt.show()

## 11. Save the model

In [ ]:
import joblib
joblib.dump({"model": best_model, "features": FEATURES, "res_lo": RES_LO, "res_hi": RES_HI,
             "current_year": CURRENT_YEAR}, "used_car_price_model.joblib")
print("Saved -> used_car_price_model.joblib")
# from google.colab import files; files.download("used_car_price_model.joblib")   # uncomment to download

## 12. 🧪 Test it yourself – prediction form

Fill in the car details and click **Predict price**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def predict_price(brand, year, km_driven, fuel, seller_type=None, transmission=None, owner=None):
    """Return (estimate, low, high) in rupees."""
    row = {"age": max(CURRENT_YEAR - int(year), 1), "km_driven": float(km_driven),
           "brand": str(brand).title(), "fuel": str(fuel).title()}
    if "seller_type" in EXTRA_CATS:   row["seller_type"] = str(seller_type).title()
    if "transmission" in EXTRA_CATS:  row["transmission"] = str(transmission).title()
    if "owner" in EXTRA_CATS:         row["owner"] = str(owner).title()

    # Unseen brand -> treat as "Other"
    if row["brand"] not in set(df["brand"]):
        row["brand"] = "Other"

    X_new = pd.DataFrame([row])[FEATURES]
    est = float(best_model.predict(X_new)[0])
    low = float(np.expm1(np.log1p(est) + RES_LO))
    high = float(np.expm1(np.log1p(est) + RES_HI))
    return est, low, high

def lakh(x): return f"₹{x/1e5:,.2f} lakh"

# ---- Widgets ----
brands = sorted(df["brand"].unique())
w_brand = widgets.Dropdown(options=brands, value="Maruti" if "Maruti" in brands else brands[0], description="Brand")
w_year  = widgets.IntSlider(value=CURRENT_YEAR-5, min=int(df["year"].min()), max=CURRENT_YEAR, description="Year")
w_km    = widgets.IntText(value=50000, description="KM driven")
w_fuel  = widgets.Dropdown(options=sorted(df["fuel"].unique()), description="Fuel")
w_extra = {}
for c, label in [("seller_type", "Seller"), ("transmission", "Gearbox"), ("owner", "Owner")]:
    if c in EXTRA_CATS:
        w_extra[c] = widgets.Dropdown(options=sorted(df[c].unique()), description=label)

btn = widgets.Button(description="Predict price", button_style="success", icon="car")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output()
        est, lo, hi = predict_price(w_brand.value, w_year.value, w_km.value, w_fuel.value,
                                    *(w_extra[c].value if c in w_extra else None
                                      for c in ["seller_type", "transmission", "owner"]))
        print(f"Estimated resale price : {lakh(est)}   (₹{est:,.0f})")
        print(f"Likely range (80%)     : {lakh(lo)}  –  {lakh(hi)}")

btn.on_click(on_click)
display(widgets.VBox([w_brand, w_year, w_km, w_fuel, *w_extra.values(), btn, out]))

### Alternative: type values in the console
Run this cell if you prefer a plain text input instead of the widget form.

In [ ]:
print("Available brands:", ", ".join(sorted(df["brand"].unique())))
print("Fuel types      :", ", ".join(sorted(df["fuel"].unique())))

brand = input("Brand (e.g. Maruti, Hyundai, Honda): ").strip()
year  = int(input(f"Manufacturing year (e.g. {CURRENT_YEAR-5}): "))
km    = float(input("Kilometres driven (e.g. 45000): "))
fuel  = input("Fuel (Petrol/Diesel/Cng/Lpg/Electric): ").strip()

def pick(col, prompt):
    opts = sorted(df[col].unique())
    return input(f"{prompt} {opts}: ").strip() if col in EXTRA_CATS else None

seller = pick("seller_type", "Seller type")
trans  = pick("transmission", "Transmission")
owner  = pick("owner", "Owner")

est, lo, hi = predict_price(brand, year, km, fuel, seller, trans, owner)
print(f"\n💰 Estimated resale price: {lakh(est)}  (₹{est:,.0f})")
print(f"   Likely range (80%)   : {lakh(lo)} – {lakh(hi)}")

## 13. Ideas to improve further
* Add engine size, mileage, power, seats (available in *Car details v3.csv*) and extract the **model name** (e.g. *Swift*, *Creta*) – the biggest accuracy boost after brand.
* Use **target encoding** or CatBoost for the car-model column (hundreds of categories).
* Try XGBoost / LightGBM / CatBoost and compare.
* Deploy as a web app with **Streamlit** or **Gradio** using the saved `.joblib` file.